In [ ]:
import sys
import time
from dataclasses import dataclass

import torch


def patch_torch_load_for_neuspell() -> None:
    if getattr(torch.load, "__name__", "") == "_torch_load_legacy":
        return

    orig_torch_load = torch.load

    def _torch_load_legacy(*args, **kwargs):
        if "weights_only" not in kwargs:
            kwargs["weights_only"] = False
        return orig_torch_load(*args, **kwargs)

    torch.load = _torch_load_legacy


patch_torch_load_for_neuspell()


from neuspell import SclstmChecker

CHECKPOINT_DIR = "../models/neuspell-scrnn-probwordnoise"

In [18]:
@dataclass
class SpellCorrector:
    checker: SclstmChecker

    @classmethod
    def load(cls) -> "SpellCorrector":
        print("load neuspell SCLSTM checker…")
        checker = SclstmChecker()
        checker.from_pretrained(CHECKPOINT_DIR)
        print("model loaded.\n")
        return cls(checker=checker)

    def correct(self, text: str) -> str:
        return self.checker.correct(text)

In [ ]:
def repl(corrector: SpellCorrector) -> None:
    print("NeuSpell SCLSTM spell correction test")
    print("enter query (empty + enter to quit).\n")

    while True:
        try:
            query = input("Query> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye.")
            return

        if not query:
            print("Quit.")
            return

        start = time.perf_counter()
        corrected = corrector.correct(query)
        elapsed_ms = (time.perf_counter() - start) * 1000

        print(f"queue: {query}")
        print(f"corrected: {corrected}")
        print(f"time: {elapsed_ms:.2f} ms")
        print("-" * 40)

In [ ]:
def main() -> None:
    corrector = SpellCorrector.load()
    repl(corrector)


if __name__ == "__main__":
    sys.exit(main())